# `softplus_nqubits.ipynb` -- annotated

**Paper:** *Fermi-Dirac machines as quantizations of neurons* (A. He, N. Liu, M. M. Wilde).

This notebook trains a **quantized neuron with a `softplus` (smooth ReLU) activation** -- **Section III.A** -- on the squared-loss function-approximation task, contributing the softplus panels of **Figure 7** (Sec. VI.D.1). It mirrors `mse_nqubits.ipynb` but swaps `tanh` for `softplus`.

| Code object | Paper |
|---|---|
| `softplus_T(x) = T*log(1+exp(x/T))` | Smooth ReLU activation, **Sec. III.A** |
| `generate_paulis(..., 'quantum'/'classical')` | TFIM **Eq. (113)** / Ising model **Eq. (114)** |
| activation observable | $\mathrm{softplus}_T(H(\omega))$ on $H(\omega)=\sum_j\omega_j H_j$ (**Eq. (16)**) |
| `dfj` / `fdd_softplus_matrix` | Derivative of the matrix smooth-ReLU function (**Theorem 6**), analogue of Theorem 1 / Eq. (20) |
| `optimize` loop | Training protocol **Sec. VI.C**, update **Eq. (119)** |

*Note:* $f'(x) = 1/(1+e^{-x/T})$ is the **sigmoid** -- softplus is its integral, exactly as in the classical smooth ReLU.

*Annotations are comments only; no executable code was changed.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import scipy 
import pandas as pd

In [ ]:
# --- Single-qubit Pauli operators (Paper Sec. II.A); building blocks of the
#     parameterized Hamiltonian H(omega)=sum_j omega_j H_j, Eq. (16). ---
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def krons(ops):
    res = ops[0]
    for op in ops[1:]:
        res = np.kron(res, op)
    return res

def to_density(vec):
    return np.outer(vec, vec.conj())

def generate_paulis(n, model="quantum"):
    """
    Generates Pauli strings for an n-qubit system.
    Includes nearest neightbor ZZ interactions, 1-body field terms, and Identity.

    Paper: term operators {H_j} for H(omega), Eq. (16). Same models as the tanh
    experiment -- "quantum" -> TFIM Eq. (113), "classical" -> IM Eq. (114) --
    but here the activation is softplus instead of tanh (Sec. III.A).
    """
    paulis = []
    
    # ZZ Interactions 
    for i in range(n-1):
        ops = [I] * n
        ops[i] = Z
        ops[i+1] = Z
        paulis.append(krons(ops))
            
    # 1-body Interactions
    for i in range(n):
        ops = [I] * n
        if model == "quantum":
            ops[i] = X
        elif model == "classical":
            ops[i] = Z
        paulis.append(krons(ops))
        
    # Identity term
    paulis.append(krons([I] * n))
    
    return paulis

def make_training_states(n):
    states = []
    dim = 2**n
    k0, k1 = np.array([1, 0]), np.array([0, 1])
    kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)

    # Computational basis states
    for bits in itertools.product([k0, k1], repeat=n):
        states.append(to_density(krons(bits)))
    
    # +/- basis states
    for bits in itertools.product([kp, km], repeat=n):
        states.append(to_density(krons(bits)))

    # GHZ state: (|00...0> + |11...1>) / sqrt(2)
    ghz_0 = krons([k0] * n)
    ghz_1 = krons([k1] * n)
    ghz = (ghz_0 + ghz_1) / np.sqrt(2)
    states.append(to_density(ghz))
    
    # Maximally mixed state
    states.append(np.eye(dim, dtype=complex) / dim)
    
    # Random mixed states
    for _ in range(3):
        A = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
        rho = A @ A.conj().T
        states.append(rho / np.trace(rho))

    return np.array(states)

# fdd_softplus_matrix: divided-difference matrix of the softplus activation
#   f(x)=T*log(1+exp(x/T)) (Paper Sec. III.A, softplus_T). F_lk=(f(l)-f(k))/(l-k)
#   with the diagonal replaced by f'(x)=sigmoid(x/T)=1/(1+exp(-x/T)). This is the
#   derivative of the matrix smooth-ReLU function (Theorem 6, derivative of the
#   smooth ReLU), used to differentiate the softplus activation observable.
def fdd_softplus_matrix(T, eigenvalues):
    l = eigenvalues.reshape(-1, 1)
    k = eigenvalues.reshape(1, -1)
    diff = l - k
    
    with np.errstate(divide='ignore', invalid='ignore'):
        f_l = T * np.logaddexp(0, l / T)
        f_k = T * np.logaddexp(0, k / T)
        res = (f_l - f_k) / diff
        
    derivative = 1.0 / (1.0 + np.exp(-l / T))
    
    mask = np.abs(diff) < 1e-10
    res = np.where(mask, derivative, res)
    
    return res

# dfj: partial derivative d/d(omega_j) Tr[ softplus_T(H(omega)) rho ]. Same
#   eigenbasis + divided-difference scheme as the tanh case (Theorem 1 /
#   Eq. (20)), specialized to the smooth-ReLU activation observable of Sec. III.A.
def dfj(rho, eigvals, eigvecs, H_j_basis, T):
    H_j_tilde = eigvecs.T.conj() @ (H_j_basis / T) @ eigvecs
    rho_tilde = eigvecs.T.conj() @ rho @ eigvecs
    F = fdd_softplus_matrix(T, eigvals)
    return np.real(np.sum(F * H_j_tilde * rho_tilde.T))

In [ ]:
# optimize: training protocol Sec. VI.C for the SOFTPLUS (smooth ReLU) neuron of
#   Sec. III.A, run for squared-loss function approximation (the softplus panels
#   of Fig. 7). Quantum (TFIM) vs classical (IM), same as the tanh notebook.
def optimize(n=3):
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    
    # Generate Hamiltonians
    pauli_q = generate_paulis(n, model="quantum")
    pauli_c = generate_paulis(n, model="classical")

    training_states = make_training_states(n)
    N_states = len(training_states)
    T = 2.0
    
    # compute target Hamiltonian and function outputs
    # Data generation (Paper Sec. VI.A): random target Hamiltonian H* (a TFIM)
    #   and true function values f(rho)=Tr[ activation(H*/T) rho ] (Eq. (120)).
    target_p = (np.random.random(len(pauli_q)) - 0.5) * 4
    H_target = sum(p * mat for p, mat in zip(target_p, pauli_q)) 
    fs = np.array([np.real(np.trace(scipy.linalg.expm(H_target / T) @ rho)) for rho in training_states])

    # initialize random parameters for quantum and classical Hamiltonian
    est_q = (np.random.random(len(pauli_q)) - 0.5)
    est_c = (np.random.random(len(pauli_c)) - 0.5)
    
    epochs = 2000  
    eta = 0.1       
    
    v_q = np.zeros(len(pauli_q))
    v_c = np.zeros(len(pauli_c))

    history_q = []
    history_c = []

    print(f"--- Running Optimization for {n} Qubits ---")
    print(f"{'Epoch':<10} | {'Quantum Loss':<15} | {'Classical Loss':<15}")
    print("-" * 45)
    
    for epoch in range(epochs):
        # --- Quantum Pass ---
        # --- Forward pass (Eq. (117)): build H_Q(omega^q), diagonalize, and apply
        #   softplus to its eigenvalues -- T*logaddexp(0, x/T) = T*log(1+exp(x/T)) --
        #   forming the smooth-ReLU activation observable softplus_T(H_Q) of
        #   Sec. III.A. Outputs f_q = Tr[ softplus_T(H_Q) rho ] (objective Eq. (17)).
        H_q = sum(p * mat for p, mat in zip(est_q, pauli_q)) / T
        eval_q, evec_q = np.linalg.eigh(H_q)
        m_softplus_q = evec_q @ np.diag(T * np.logaddexp(0, eval_q / T)) @ evec_q.T.conj()
        fq_q = np.array([np.real(np.trace(m_softplus_q @ rho)) for rho in training_states])
        
        grad_q = np.zeros(len(pauli_q))
        for j in range(len(pauli_q)):
            g_j = sum(2 * (fq_q[i] - fs[i]) * dfj(training_states[i], eval_q, evec_q, pauli_q[j], T) for i in range(N_states))
            grad_q[j] = g_j / N_states
            
        # Gradient-descent update (Paper Eq. (119)); inner derivative from dfj
        #   (Theorem 6 for the smooth-ReLU activation).
        v_q = eta * grad_q
        est_q -= v_q
        
        l_q = np.mean(np.square(fq_q - fs))
        history_q.append(l_q)

        # --- Classical pass: softplus neuron on the commuting Ising model H_C
        #   (Eq. (114)) -> reduces to a classical neuron (Sec. II.A). ---
        H_c = sum(p * mat for p, mat in zip(est_c, pauli_c)) / T
        eval_c, evec_c = np.linalg.eigh(H_c)
        m_softplus_c = evec_c @ np.diag(T * np.logaddexp(0, eval_c / T)) @ evec_c.T.conj()
        fq_c = np.array([np.real(np.trace(m_softplus_c @ rho)) for rho in training_states])
        
        grad_c = np.zeros(len(pauli_c))
        for j in range(len(pauli_c)):
            g_j = sum(2 * (fq_c[i] - fs[i]) * dfj(training_states[i], eval_c, evec_c, pauli_c[j], T) for i in range(N_states))
            grad_c[j] = g_j / N_states
            
        v_c = eta * grad_c
        est_c -= v_c

        l_c = np.mean(np.square(fq_c - fs))
        history_c.append(l_c)

        epoch_data = {
            "Epoch": epoch,
            "Quantum_Loss": l_q,
            "Classical_Loss": l_c,
        }

        if epoch % 20 == 0:
            print(f"{epoch:<10} | {l_q:<15.8f} | {l_c:<15.8f}")
        metrics_log.append(epoch_data)

    print("\n--- Final Results ---")
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/mse_softplus_{n}qubit_ising.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print("Target Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c;

In [ ]:
n = 7
history_q, history_c = optimize(n);

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import FuncFormatter, NullFormatter
# plot: softplus panels of Paper Fig. 7 -- squared loss vs epoch, TFIM vs IM.
def plot(history_q, history_c, n):
    plt.figure(figsize=(10, 6))
    plt.plot(history_c, label=r'Classical Model ($H_\text{IM}$)', color='red', linewidth=2, linestyle='--')
    plt.plot(history_q, label=r'Quantum Model ($H_\text{TFIM}$)', color='blue', linewidth=2)
    plt.xlabel('Epoch', fontsize=24)
    plt.ylabel('Squared Loss', fontsize=24)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)    
    plt.title(f'{n} Qubits, TFIM, softplus', fontsize=24)
    plt.legend(fontsize=24)
    # plt.yscale('log')

    plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    plt.subplots_adjust(left=0.15, right=0.97, bottom=0.18, top=0.90)
    plt.savefig(f"plots/mse_softplus_{n}qubit_ising.pdf", format="pdf")
    plt.show()

In [ ]:
plot(history_q, history_c, n)

In [ ]:
n = 2 
csv_filename = f"plots/mse_softplus_{n}qubit_ising.csv"
df = pd.read_csv(csv_filename)

history_q = df['Quantum_Loss'].values
history_c = df['Classical_Loss'].values

plot(history_q, history_c, n)